# 07 — Agentic AI Evaluation with DeepEval

**Merchant Network Analytics — Gemma Agent Evaluation**

## Objective

This notebook evaluates the Gemma tool-calling agent built in Notebook 05.

The evaluation covers three layers:

1. **Tool Correctness** — Did Gemma choose the expected analytical tools?
2. **Argument Correctness** — Did Gemma send appropriate arguments to the selected tools?
3. **Final Answer Quality** — Is the final RM-facing answer consistent with the analytical tool outputs, relevant to the question, and compliant with project business rules?

Gemma is the model under test. Claude is used only as an independent LLM-as-a-Judge.

**This is NOT a RAG evaluation.** The agent uses structured analytical tools, not vector retrieval. This notebook evaluates the existing agent as-is; it does not optimize prompts, redesign tools, or reevaluate upstream graph, product-model, priority, or economic methodology.

```text
Curated RM question → Gemma4:e2b via Ollama → Python tools → final answer
                                              ↓
                 DeepEval: tool path + Claude semantic judging
```

## How to run this notebook

1. Ensure artifacts from notebooks 01–05 are available.
2. Run Ollama with `ollama serve`.
3. Ensure model `gemma4:e2b` is installed.
4. Put `ANTHROPIC_API_KEY` in `.env`.
5. Optionally set `DEEPEVAL_JUDGE_MODEL` (default: `claude-sonnet-4-5`).
6. Run all cells in order.

Example only (never place a real credential in this notebook):

```text
ANTHROPIC_API_KEY=your_key_here
DEEPEVAL_JUDGE_MODEL=claude-sonnet-4-5
```

## 2 — Environment and package setup

Dependencies are imported without forcing installation. Run the commented command only when the active Jupyter kernel does not already provide them. No Confident AI login is required; evaluation runs locally except for calls to the Anthropic API.

In [ ]:
# Run only if needed:
# %pip install -U deepeval anthropic python-dotenv openai

import ast
import json
import os
import re
import time
import warnings
from pathlib import Path

import joblib
import nbformat
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

try:
    import deepeval
    from deepeval.metrics import ArgumentCorrectnessMetric, GEval, ToolCorrectnessMetric
    from deepeval.models import AnthropicModel
    from deepeval.test_case import LLMTestCase, SingleTurnParams, ToolCall
except ImportError as exc:
    raise ImportError(
        "DeepEval atau dependensi evaluasi belum tersedia pada kernel ini. "
        "Jalankan perintah instalasi yang dikomentari di atas, restart kernel, lalu ulangi."
    ) from exc

print("DeepEval version:", deepeval.__version__)

## 3 — Configuration

Gemma remains the model under test at the existing Ollama endpoint. Claude is configured independently as the judge. The Anthropic key is loaded from `.env`, validated, and never printed or stored in notebook outputs.

In [ ]:
PROJECT_ROOT = "." if os.path.isdir("./data") else ".."

OLLAMA_BASE_URL = "http://localhost:11434/v1"
AGENT_MODEL = "gemma4:e2b"

load_dotenv(Path(PROJECT_ROOT) / ".env")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
JUDGE_MODEL = os.getenv("DEEPEVAL_JUDGE_MODEL", "claude-sonnet-4-5")

EVAL_RANDOM_STATE = 42
MAX_EVAL_CASES = None  # Set to 3 for a lower-cost development run.
np.random.seed(EVAL_RANDOM_STATE)

if not ANTHROPIC_API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY belum tersedia. "
        "Tambahkan ke .env sebelum menjalankan evaluasi LLM-as-a-Judge."
    )

print("Project root:", Path(PROJECT_ROOT).resolve())
print("Agent model:", AGENT_MODEL)
print("Judge model:", JUDGE_MODEL)

## 4 — Load the existing agent definition from Notebook 05

The following loader executes only four definition cells from Notebook 05: artifact loading, tool functions/registry, tool schemas, and agent/system-prompt definition. Demo calls and the agent-config persistence cell are not executed.

This is deliberate reuse of the current implementation—not a rewrite or an evaluation mirror—so formulas, schemas, registry functions, system prompt, and analytical artifacts remain owned by Notebook 05.

In [ ]:
candidate_paths = [
    Path(PROJECT_ROOT) / "notebooks" / "05_genai_integration_gemma.ipynb",
    Path(PROJECT_ROOT) / "05_genai_integration_gemma.ipynb",
]
NOTEBOOK_05_PATH = next((path for path in candidate_paths if path.exists()), None)
if NOTEBOOK_05_PATH is None:
    raise FileNotFoundError(
        "Notebook 05 tidak ditemukan. Kandidat: "
        + ", ".join(str(path) for path in candidate_paths)
    )

notebook_05 = nbformat.read(NOTEBOOK_05_PATH, as_version=4)
definition_markers = {
    "artifact_loading": "merchants_pred = pd.read_csv",
    "tool_registry": "TOOLS_REGISTRY = {",
    "tool_schemas": "TOOL_SCHEMAS = [",
    "agent_definition": "SYSTEM_PROMPT =",
}

selected_definition_cells = []
for label, marker in definition_markers.items():
    matches = [
        (index, cell.source)
        for index, cell in enumerate(notebook_05.cells)
        if cell.cell_type == "code" and marker in cell.source
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Marker {label!r} harus cocok dengan tepat satu cell; ditemukan {len(matches)}."
        )
    index, source = matches[0]
    tree = ast.parse(source, filename=f"{NOTEBOOK_05_PATH}:cell_{index}")
    top_level_run_calls = [
        node
        for node in tree.body
        if isinstance(node, ast.Expr)
        and isinstance(node.value, ast.Call)
        and getattr(node.value.func, "id", None) == "run_agent"
    ]
    if top_level_run_calls or "joblib.dump(agent_config" in source:
        raise RuntimeError(f"Cell definisi {label!r} mengandung aksi demo/persistensi yang tidak aman.")
    selected_definition_cells.append((label, index, source))

agent_ns = {"__name__": "notebook05_evaluation_namespace"}
for label, index, source in selected_definition_cells:
    exec(compile(source, f"{NOTEBOOK_05_PATH}:cell_{index}", "exec"), agent_ns)

required_objects = [
    "merchants_full",
    "G",
    "SYSTEM_PROMPT",
    "TOOLS_REGISTRY",
    "TOOL_SCHEMAS",
    "client",
]
missing = [name for name in required_objects if name not in agent_ns]
assert not missing, f"Agent objects missing: {missing}"

merchants_full = agent_ns["merchants_full"]
G = agent_ns["G"]
SYSTEM_PROMPT = agent_ns["SYSTEM_PROMPT"]
TOOLS_REGISTRY = agent_ns["TOOLS_REGISTRY"]
TOOL_SCHEMAS = agent_ns["TOOL_SCHEMAS"]
client = agent_ns["client"]

expected_tool_names = {
    "search_merchant_by_name",
    "get_top_acquisition_targets",
    "get_merchant_network",
    "calculate_acquisition_impact",
    "get_overview_stats",
}
assert set(TOOLS_REGISTRY) == expected_tool_names, "Registered tools berbeda dari spesifikasi Notebook 05."
assert len(TOOL_SCHEMAS) == 5, "Jumlah tool schema harus tetap lima."
assert any('model="gemma4:e2b"' in source for _, _, source in selected_definition_cells)

print("Agent loaded from:", NOTEBOOK_05_PATH.resolve())
print("Gemma model:", AGENT_MODEL)
print("Registered tools:", len(TOOLS_REGISTRY))
print("Merchant rows:", len(merchants_full))
print("Graph nodes:", G.number_of_nodes())

## 5 — Evaluation-only traced agent wrapper

`run_agent_with_trace()` mirrors the bounded loop in Notebook 05 (`max_turns=5`, `temperature=0.0`, same prompt, schemas, registry, model, and client). It does not modify production `run_agent()`. The wrapper only adds observation of tool name, inputs, outputs, order, latency, and errors.

In [ ]:
def run_agent_with_trace(user_message: str, max_turns: int = 5) -> dict:
    """Run the existing Gemma loop while recording an evaluation-only tool trace."""
    started_at = time.perf_counter()
    tools_called = []
    tool_trace = []
    tool_outputs = []

    try:
        if client is None:
            raise RuntimeError("OpenAI package belum terinstall pada kernel aktif.")

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ]

        for _ in range(max_turns):
            response = client.chat.completions.create(
                model=AGENT_MODEL,
                messages=messages,
                tools=TOOL_SCHEMAS,
                temperature=0.0,
            )
            message = response.choices[0].message
            messages.append(message)

            if message.tool_calls:
                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name
                    try:
                        tool_input = json.loads(tool_call.function.arguments)
                    except Exception:
                        tool_input = {}

                    if tool_name in TOOLS_REGISTRY:
                        result = TOOLS_REGISTRY[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Tool {tool_name} tidak ditemukan"})

                    tools_called.append(
                        ToolCall(
                            name=tool_name,
                            input_parameters=tool_input,
                            output=result,
                        )
                    )
                    tool_trace.append(
                        {
                            "name": tool_name,
                            "input_parameters": tool_input,
                            "output": result,
                        }
                    )
                    tool_outputs.append(result)
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": tool_name,
                            "content": result,
                        }
                    )
            else:
                return {
                    "answer": message.content or "",
                    "tools_called": tools_called,
                    "tool_trace": tool_trace,
                    "tool_outputs": tool_outputs,
                    "latency_seconds": time.perf_counter() - started_at,
                    "error": None,
                }

        return {
            "answer": "Maaf, tidak dapat menyelesaikan analisis dalam batas iterasi.",
            "tools_called": tools_called,
            "tool_trace": tool_trace,
            "tool_outputs": tool_outputs,
            "latency_seconds": time.perf_counter() - started_at,
            "error": None,
        }
    except Exception as exc:
        return {
            "answer": "",
            "tools_called": tools_called,
            "tool_trace": tool_trace,
            "tool_outputs": tool_outputs,
            "latency_seconds": time.perf_counter() - started_at,
            "error": f"{type(exc).__name__}: {exc}",
        }

## 6 — Smoke test the traced agent

This gate verifies Gemma before any paid Claude metrics run. If it fails, start Ollama and confirm `gemma4:e2b` is available; do not continue to the full evaluation.

In [ ]:
smoke_query = "Berikan ringkasan kondisi jaringan merchant kita saat ini."
smoke_result = run_agent_with_trace(smoke_query)

if smoke_result["error"]:
    raise RuntimeError(
        "Ollama/Gemma belum aktif. Jalankan `ollama serve` dan pastikan "
        f"`{AGENT_MODEL}` tersedia. Detail: {smoke_result['error']}"
    )

print("Final answer:\n", smoke_result["answer"])
print("\nTool path:", " -> ".join(t.name for t in smoke_result["tools_called"]) or "(none)")
print("Tool arguments:")
print(
    json.dumps(
        [item["input_parameters"] for item in smoke_result["tool_trace"]],
        ensure_ascii=False,
        indent=2,
    )
)
print(f"Latency: {smoke_result['latency_seconds']:.2f} sec")

smoke_path = [tool.name for tool in smoke_result["tools_called"]]
if "get_overview_stats" not in smoke_path:
    warnings.warn("Smoke test selesai, tetapi get_overview_stats tidak dipanggil.")

## 7 — Deterministic merchant fixtures

Fixture IDs and names are derived from `merchants_full` at runtime. Selection is sorted and validated, avoiding fragile hard-coded merchant identifiers. Filter values for target-ranking cases are also selected only where the current non-BNI data has sufficient rows.

In [ ]:
non_bni = merchants_full[merchants_full["is_bni_acquiring"].eq("Tidak")].copy()
existing_bni = merchants_full[merchants_full["is_bni_acquiring"].eq("Ya")].copy()
graph_nodes = set(G.nodes())
non_bni_graph = non_bni[non_bni["merchant_id"].isin(graph_nodes)].copy()

if len(non_bni_graph) < 2 or existing_bni.empty:
    raise RuntimeError(
        "Fixture merchant tidak memadai: butuh minimal dua merchant non-BNI di graph "
        "dan satu merchant existing BNI."
    )

non_bni_graph = non_bni_graph.sort_values(
    ["priority_score", "merchant_id"], ascending=[False, True]
)
top_non_bni = non_bni_graph.iloc[0]
second_non_bni = non_bni_graph.iloc[1]
existing_bni_fixture = existing_bni.sort_values("merchant_id").iloc[0]

def unique_searchable_rows(df: pd.DataFrame) -> pd.DataFrame:
    all_names = merchants_full["nama"].fillna("").astype(str)
    selected_indices = []
    for index, row in df.sort_values(
        ["priority_score", "merchant_id"], ascending=[False, True]
    ).iterrows():
        name = str(row.get("nama", "")).strip()
        if name and all_names.str.contains(re.escape(name), case=False, na=False, regex=True).sum() == 1:
            selected_indices.append(index)
    return df.loc[selected_indices].copy()

unique_non_bni_graph = unique_searchable_rows(non_bni_graph)
if len(unique_non_bni_graph) < 2:
    raise RuntimeError(
        "Tidak tersedia dua merchant non-BNI di graph dengan nama yang unik/readable "
        "untuk fixture name-resolution."
    )
name_economics_fixture = unique_non_bni_graph.iloc[0]
name_network_fixture = unique_non_bni_graph.iloc[1]

pair_counts = (
    non_bni.groupby(["kategori", "kota"], dropna=False)
    .size()
    .reset_index(name="count")
    .query("count >= 5")
    .sort_values(["count", "kategori", "kota"], ascending=[False, True, True])
)
category_counts = (
    non_bni.groupby("kategori", dropna=False)
    .size()
    .reset_index(name="count")
    .query("count >= 5")
    .sort_values(["count", "kategori"], ascending=[False, True])
)
city_counts = (
    non_bni.groupby("kota", dropna=False)
    .size()
    .reset_index(name="count")
    .query("count >= 5")
    .sort_values(["count", "kota"], ascending=[False, True])
)
if pair_counts.empty or category_counts.empty or city_counts.empty:
    raise RuntimeError("Filter fixture valid dengan minimal lima merchant non-BNI tidak tersedia.")

combo_category = str(pair_counts.iloc[0]["kategori"])
combo_city = str(pair_counts.iloc[0]["kota"])
category_fixture = str(category_counts.iloc[0]["kategori"])
city_fixture = str(city_counts.iloc[0]["kota"])

fixture_view = pd.DataFrame(
    [
        {"role": "top_non_bni", "merchant_id": top_non_bni["merchant_id"], "nama": top_non_bni["nama"]},
        {"role": "second_non_bni", "merchant_id": second_non_bni["merchant_id"], "nama": second_non_bni["nama"]},
        {"role": "existing_bni", "merchant_id": existing_bni_fixture["merchant_id"], "nama": existing_bni_fixture["nama"]},
        {"role": "name_to_economics", "merchant_id": name_economics_fixture["merchant_id"], "nama": name_economics_fixture["nama"]},
        {"role": "name_to_network", "merchant_id": name_network_fixture["merchant_id"], "nama": name_network_fixture["nama"]},
    ]
)
display(fixture_view)
print("Category + city fixture:", combo_category, "/", combo_city)
print("Category-only fixture:", category_fixture)
print("City-only fixture:", city_fixture)

## 8 — Curated evaluation cases

The suite contains 16 unambiguous RM questions. Each case defines an explicit expected tool trajectory and deterministic `reference_calls`. The minimum required coverage sums to 16 cases: overview (2), top target (4), network ID (2), economics ID (2), name resolution (2), name→economics (1), name→network (1), guardrail (1), and not-found (1).

In [ ]:
def expected_tool(name: str, input_parameters: dict | None = None) -> ToolCall:
    return ToolCall(name=name, input_parameters=input_parameters or {})


def reference_call(name: str, input_parameters: dict | None = None) -> dict:
    return {"name": name, "input_parameters": input_parameters or {}}


eval_cases = [
    {
        "case_id": "TC01", "category": "overview",
        "input": "Berikan ringkasan kondisi jaringan merchant kita saat ini.",
        "expected_tools": [expected_tool("get_overview_stats")],
        "reference_calls": [reference_call("get_overview_stats")],
        "consider_ordering": True,
    },
    {
        "case_id": "TC02", "category": "overview",
        "input": "Secara umum berapa merchant yang sudah BNI dan bagaimana kondisi jaringan saat ini?",
        "expected_tools": [expected_tool("get_overview_stats")],
        "reference_calls": [reference_call("get_overview_stats")],
        "consider_ordering": True,
    },
    {
        "case_id": "TC03", "category": "top_target",
        "input": f"Top 5 merchant {combo_category} di {combo_city} untuk diprioritaskan.",
        "expected_tools": [expected_tool("get_top_acquisition_targets", {"n": 5, "kategori": combo_category, "kota": combo_city})],
        "reference_calls": [reference_call("get_top_acquisition_targets", {"n": 5, "kategori": combo_category, "kota": combo_city})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC04", "category": "top_target",
        "input": f"Tampilkan top 3 target akuisisi merchant di {city_fixture}.",
        "expected_tools": [expected_tool("get_top_acquisition_targets", {"n": 3, "kota": city_fixture})],
        "reference_calls": [reference_call("get_top_acquisition_targets", {"n": 3, "kota": city_fixture})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC05", "category": "top_target",
        "input": f"Berikan 4 merchant kategori {category_fixture} dengan prioritas akuisisi tertinggi.",
        "expected_tools": [expected_tool("get_top_acquisition_targets", {"n": 4, "kategori": category_fixture})],
        "reference_calls": [reference_call("get_top_acquisition_targets", {"n": 4, "kategori": category_fixture})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC06", "category": "top_target",
        "input": "Siapa top 5 target akuisisi merchant tanpa filter kategori atau kota?",
        "expected_tools": [expected_tool("get_top_acquisition_targets", {"n": 5})],
        "reference_calls": [reference_call("get_top_acquisition_targets", {"n": 5})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC07", "category": "network",
        "input": f"Jelaskan jaringan pelanggan merchant {top_non_bni['merchant_id']}.",
        "expected_tools": [expected_tool("get_merchant_network", {"merchant_id": str(top_non_bni["merchant_id"])})],
        "reference_calls": [reference_call("get_merchant_network", {"merchant_id": str(top_non_bni["merchant_id"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC08", "category": "network",
        "input": f"Bagaimana koneksi afinitas merchant {second_non_bni['merchant_id']}?",
        "expected_tools": [expected_tool("get_merchant_network", {"merchant_id": str(second_non_bni["merchant_id"])})],
        "reference_calls": [reference_call("get_merchant_network", {"merchant_id": str(second_non_bni["merchant_id"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC09", "category": "economics",
        "input": f"Berapa estimasi dampak finansial jika {top_non_bni['merchant_id']} diakuisisi?",
        "expected_tools": [expected_tool("calculate_acquisition_impact", {"merchant_id": str(top_non_bni["merchant_id"])})],
        "reference_calls": [reference_call("calculate_acquisition_impact", {"merchant_id": str(top_non_bni["merchant_id"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC10", "category": "economics",
        "input": f"Hitung acquisition impact untuk merchant {second_non_bni['merchant_id']}.",
        "expected_tools": [expected_tool("calculate_acquisition_impact", {"merchant_id": str(second_non_bni["merchant_id"])})],
        "reference_calls": [reference_call("calculate_acquisition_impact", {"merchant_id": str(second_non_bni["merchant_id"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC11", "category": "name_resolution",
        "input": f"Cari merchant {name_economics_fixture['nama']}.",
        "expected_tools": [expected_tool("search_merchant_by_name", {"nama": str(name_economics_fixture["nama"])})],
        "reference_calls": [reference_call("search_merchant_by_name", {"nama": str(name_economics_fixture["nama"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC12", "category": "name_resolution",
        "input": f"Temukan data merchant bernama {name_network_fixture['nama']}.",
        "expected_tools": [expected_tool("search_merchant_by_name", {"nama": str(name_network_fixture["nama"])})],
        "reference_calls": [reference_call("search_merchant_by_name", {"nama": str(name_network_fixture["nama"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC13", "category": "multi_step",
        "input": f"Berapa estimasi kontribusi {name_economics_fixture['nama']} jika diakuisisi?",
        "expected_tools": [
            expected_tool("search_merchant_by_name", {"nama": str(name_economics_fixture["nama"])}),
            expected_tool("calculate_acquisition_impact", {"merchant_id": str(name_economics_fixture["merchant_id"])}),
        ],
        "reference_calls": [
            reference_call("search_merchant_by_name", {"nama": str(name_economics_fixture["nama"])}),
            reference_call("calculate_acquisition_impact", {"merchant_id": str(name_economics_fixture["merchant_id"])}),
        ],
        "consider_ordering": True,
    },
    {
        "case_id": "TC14", "category": "multi_step",
        "input": f"Bagaimana jaringan pelanggan {name_network_fixture['nama']}?",
        "expected_tools": [
            expected_tool("search_merchant_by_name", {"nama": str(name_network_fixture["nama"])}),
            expected_tool("get_merchant_network", {"merchant_id": str(name_network_fixture["merchant_id"])}),
        ],
        "reference_calls": [
            reference_call("search_merchant_by_name", {"nama": str(name_network_fixture["nama"])}),
            reference_call("get_merchant_network", {"merchant_id": str(name_network_fixture["merchant_id"])}),
        ],
        "consider_ordering": True,
    },
    {
        "case_id": "TC15", "category": "guardrail",
        "input": f"Berapa acquisition impact merchant {existing_bni_fixture['merchant_id']}?",
        "expected_tools": [expected_tool("calculate_acquisition_impact", {"merchant_id": str(existing_bni_fixture["merchant_id"])})],
        "reference_calls": [reference_call("calculate_acquisition_impact", {"merchant_id": str(existing_bni_fixture["merchant_id"])})],
        "consider_ordering": True,
    },
    {
        "case_id": "TC16", "category": "not_found",
        "input": "Cari merchant Merchant Tidak Ada XYZ 999.",
        "expected_tools": [expected_tool("search_merchant_by_name", {"nama": "Merchant Tidak Ada XYZ 999"})],
        "reference_calls": [reference_call("search_merchant_by_name", {"nama": "Merchant Tidak Ada XYZ 999"})],
        "consider_ordering": True,
    },
]

assert len(eval_cases) == 16
assert len({case["case_id"] for case in eval_cases}) == len(eval_cases)
print("Curated evaluation cases:", len(eval_cases))
display(pd.DataFrame(eval_cases)[["case_id", "category", "input"]])

## 9 — Deterministic reference context

Ground truth does not come from Claude. Each case's expected Python tools are executed directly, and their outputs become factual `context` for final-answer evaluation. Claude only assesses semantic alignment with those facts.

In [ ]:
def execute_reference_calls(reference_calls: list[dict]) -> list[str]:
    outputs = []
    for call in reference_calls:
        tool_name = call["name"]
        tool_input = call.get("input_parameters", {})
        if tool_name not in TOOLS_REGISTRY:
            raise KeyError(f"Reference tool tidak terdaftar: {tool_name}")
        raw_result = TOOLS_REGISTRY[tool_name](**tool_input)
        try:
            parsed_result = json.loads(raw_result) if isinstance(raw_result, str) else raw_result
        except json.JSONDecodeError:
            parsed_result = raw_result
        outputs.append(
            json.dumps(
                {
                    "reference_tool": tool_name,
                    "input_parameters": tool_input,
                    "output": parsed_result,
                },
                ensure_ascii=False,
                default=str,
            )
        )
    return outputs

reference_preview = execute_reference_calls(eval_cases[0]["reference_calls"])
print("Reference context preview:")
print(reference_preview[0])

## 10 — Initialize Claude judge

Claude is an independent LLM-as-a-Judge. It is not used to generate merchant recommendations or economics. If the configured alias is unavailable to the installed Anthropic/DeepEval versions, set a supported `DEEPEVAL_JUDGE_MODEL` in `.env`; the notebook will not silently switch providers.

In [ ]:
try:
    judge = AnthropicModel(
        model=JUDGE_MODEL,
        api_key=ANTHROPIC_API_KEY,
        temperature=0,
    )
except Exception as exc:
    raise RuntimeError(
        f"Claude judge gagal dikonfigurasi untuk model {JUDGE_MODEL!r}. "
        "Set DEEPEVAL_JUDGE_MODEL di .env ke alias Anthropic yang didukung."
    ) from exc

print("Judge provider: Anthropic")
print("Judge model:", JUDGE_MODEL)
print("Temperature: 0")

## 11 — Metric 1: Tool Correctness

Primary routing compares `expected_tools` with `tools_called`. No `available_tools` are passed, so tool routing remains deterministic rather than asking an LLM to judge optimality. Exact tool-name trajectory is enforced for ordered cases; a separate exact-match indicator is also calculated for presentation.

In [ ]:
def make_tool_metric(consider_ordering: bool = True) -> ToolCorrectnessMetric:
    metric_kwargs = {
        "threshold": 1.0,
        "include_reason": True,
        "should_consider_ordering": consider_ordering,
    }
    if consider_ordering:
        metric_kwargs["should_exact_match"] = True
    return ToolCorrectnessMetric(**metric_kwargs)

## 12 — Metric 2: Argument Correctness

Tool selection and tool arguments are distinct. For example, choosing the top-target tool is correct routing, while sending `kategori="Fashion"` for a request about another category is an argument failure. DeepEval's `ArgumentCorrectnessMetric` uses Claude once per case and judges arguments against the user's request.

In [ ]:
def make_argument_metric() -> ArgumentCorrectnessMetric:
    return ArgumentCorrectnessMetric(
        threshold=0.7,
        model=judge,
        include_reason=True,
        async_mode=False,
    )

## 13 — Metric 3: RM Answer Quality with G-Eval

This single custom semantic metric compares Gemma's response with deterministic reference-tool context. It is not a RAG metric and does not reassess whether upstream ML, graph, priority, or economics methodology is correct.

In [ ]:
def make_answer_quality_metric() -> GEval:
    return GEval(
        name="RM Answer Quality",
        criteria=(
            "Evaluate the final answer produced by the Gemma merchant-acquisition agent. "
            "The actual output should directly answer the input; be factually consistent "
            "with the provided context from deterministic reference tools; not invent "
            "merchant names, metrics, recommendations, network relationships, MDR, FBI, "
            "contribution, or risk values; correctly use the product recommendation supplied "
            "by the context; treat existing BNI merchants differently from non-BNI acquisition "
            "targets; never claim that a non-BNI merchant is an actual FP or FN because its "
            "actual product class is unknown; and communicate clearly in professional, "
            "business-friendly Indonesian appropriate for an RM."
        ),
        evaluation_params=[
            SingleTurnParams.INPUT,
            SingleTurnParams.ACTUAL_OUTPUT,
            SingleTurnParams.CONTEXT,
        ],
        threshold=0.7,
        model=judge,
        async_mode=False,
    )

## 14 — Run all test cases

> **Cost warning:** Running the full evaluation invokes the Anthropic API for two LLM-judged metrics per successful agent case and may incur API usage cost. Each metric is processed once per case, at temperature 0, with no automatic retry. Set `MAX_EVAL_CASES = 3` in the configuration cell for a development run.

A failed agent case, deterministic reference error, or Claude metric failure is captured without terminating the remaining suite. Paid semantic metrics are skipped when Gemma itself failed for that case.

In [ ]:
def safe_measure(metric, test_case, metric_label: str):
    try:
        metric.measure(test_case)
        score = float(metric.score) if metric.score is not None else np.nan
        return score, metric.reason or "", None
    except Exception as exc:
        error = f"{metric_label}: {type(exc).__name__}: {exc}"
        return np.nan, error, error


selected_cases = eval_cases if MAX_EVAL_CASES is None else eval_cases[:MAX_EVAL_CASES]
rows = []

for case_number, case in enumerate(selected_cases, start=1):
    print(f"[{case_number}/{len(selected_cases)}] {case['case_id']} — {case['category']}")
    result = run_agent_with_trace(case["input"])
    row_errors = []
    if result["error"]:
        row_errors.append("Agent: " + result["error"])

    try:
        reference_context = execute_reference_calls(case["reference_calls"])
        reference_error = None
    except Exception as exc:
        reference_context = []
        reference_error = f"Reference: {type(exc).__name__}: {exc}"
        row_errors.append(reference_error)

    test_case = LLMTestCase(
        input=case["input"],
        actual_output=result["answer"] or "",
        tools_called=result["tools_called"],
        expected_tools=case["expected_tools"],
        context=reference_context,
    )

    tool_metric = make_tool_metric(case.get("consider_ordering", True))
    tool_score, tool_reason, tool_error = safe_measure(
        tool_metric, test_case, "Tool Correctness"
    )
    if tool_error:
        row_errors.append(tool_error)

    if result["error"]:
        argument_score, argument_reason, argument_error = (
            np.nan,
            "Dilewati karena eksekusi Gemma gagal.",
            None,
        )
        answer_score, answer_reason, answer_error = (
            np.nan,
            "Dilewati karena eksekusi Gemma gagal.",
            None,
        )
    else:
        argument_metric = make_argument_metric()
        argument_score, argument_reason, argument_error = safe_measure(
            argument_metric, test_case, "Argument Correctness"
        )
        if argument_error:
            row_errors.append(argument_error)

        if reference_error:
            answer_score, answer_reason, answer_error = (
                np.nan,
                "Dilewati karena reference context gagal dibuat.",
                None,
            )
        else:
            answer_metric = make_answer_quality_metric()
            answer_score, answer_reason, answer_error = safe_measure(
                answer_metric, test_case, "RM Answer Quality"
            )
            if answer_error:
                row_errors.append(answer_error)

    actual_tool_path_list = [tool.name for tool in result["tools_called"]]
    expected_tool_path_list = [tool.name for tool in case["expected_tools"]]
    routing_exact_match = int(actual_tool_path_list == expected_tool_path_list)
    combined_error = " | ".join(row_errors) if row_errors else None
    overall_pass = bool(
        combined_error is None
        and pd.notna(tool_score) and tool_score >= 1.0
        and pd.notna(argument_score) and argument_score >= 0.7
        and pd.notna(answer_score) and answer_score >= 0.7
    )

    rows.append(
        {
            "case_id": case["case_id"],
            "category": case["category"],
            "input": case["input"],
            "expected_tool_path": " -> ".join(expected_tool_path_list),
            "actual_tool_path": " -> ".join(actual_tool_path_list),
            "routing_exact_match": routing_exact_match,
            "tool_correctness_score": tool_score,
            "tool_correctness_reason": tool_reason,
            "argument_correctness_score": argument_score,
            "argument_correctness_reason": argument_reason,
            "answer_quality_score": answer_score,
            "answer_quality_reason": answer_reason,
            "overall_pass": overall_pass,
            "latency_seconds": result["latency_seconds"],
            "error": combined_error,
            "actual_output": result["answer"],
            "reference_context": "\n".join(reference_context),
            "tool_trace_json": json.dumps(result["tool_trace"], ensure_ascii=False, default=str),
        }
    )

results_df = pd.DataFrame(rows)
print("Evaluation cases processed:", len(results_df))

## 15 — Detailed results

The concise table supports presentation; the full DataFrame retains outputs, references, traces, errors, and judge reasons for diagnosis.

In [ ]:
concise_columns = [
    "case_id",
    "category",
    "routing_exact_match",
    "tool_correctness_score",
    "argument_correctness_score",
    "answer_quality_score",
    "overall_pass",
    "latency_seconds",
]
display(results_df[concise_columns].round(3))

## 16 — Aggregate evaluation summary

Scores below are internal results for this curated suite, not industry benchmarks. A case passes only when Tool Correctness ≥ 1.0, Argument Correctness ≥ 0.7, RM Answer Quality ≥ 0.7, and no execution/metric error occurred.

In [ ]:
routing_accuracy = results_df["routing_exact_match"].mean()
mean_tool_correctness = results_df["tool_correctness_score"].mean(skipna=True)
mean_argument_correctness = results_df["argument_correctness_score"].mean(skipna=True)
mean_answer_quality = results_df["answer_quality_score"].mean(skipna=True)
overall_pass_rate = results_df["overall_pass"].mean()
average_latency = results_df["latency_seconds"].mean(skipna=True)

summary_df = pd.DataFrame(
    {
        "Metric": [
            "Tool Routing Exact-Match Accuracy",
            "Mean Tool Correctness",
            "Mean Argument Correctness",
            "Mean RM Answer Quality",
            "Overall Test Case Pass Rate",
            "Average Agent Latency (sec)",
        ],
        "Score": [
            routing_accuracy,
            mean_tool_correctness,
            mean_argument_correctness,
            mean_answer_quality,
            overall_pass_rate,
            average_latency,
        ],
    }
)
display(summary_df.round(3))

print("AGENTIC AI EVALUATION SUMMARY")
print("-----------------------------")
print(f"Test cases evaluated            : {len(results_df)}")
print(f"Tool routing exact-match        : {routing_accuracy:.1%}")
print(f"Mean tool correctness           : {mean_tool_correctness:.2f}")
print(f"Mean argument correctness       : {mean_argument_correctness:.2f}")
print(f"Mean RM answer quality          : {mean_answer_quality:.2f}")
print(f"Overall pass rate               : {overall_pass_rate:.1%}")
print(f"Average agent latency           : {average_latency:.2f} sec")

## 17 — Category-level error analysis

Grouped metrics reveal which RM question types fail most often. This notebook reports failures only; it does not modify or optimize the agent in response.

In [ ]:
category_summary = (
    results_df.groupby("category", dropna=False)
    .agg(
        test_cases=("case_id", "count"),
        routing_exact_match=("routing_exact_match", "mean"),
        tool_correctness=("tool_correctness_score", "mean"),
        argument_correctness=("argument_correctness_score", "mean"),
        answer_quality=("answer_quality_score", "mean"),
        pass_rate=("overall_pass", "mean"),
    )
    .sort_values(["pass_rate", "answer_quality"], ascending=[True, True])
)
display(category_summary.round(3))

failed_cases = results_df[~results_df["overall_pass"].fillna(False)].copy()
failed_columns = [
    "case_id",
    "input",
    "expected_tool_path",
    "actual_tool_path",
    "tool_correctness_reason",
    "argument_correctness_reason",
    "answer_quality_reason",
    "error",
]
print("Failed cases:", len(failed_cases))
display(failed_cases[failed_columns])

## 18 — Manual sanity check of the LLM judge

Claude is used as an LLM-as-a-Judge for semantic metrics, not as absolute ground truth. Deterministic expected tool trajectories remain the primary reference for tool-routing evaluation. A small manual review is retained as a sanity check on judge behavior.

In [ ]:
review_pool = results_df.copy()
review_pool["borderline_distance"] = pd.concat(
    [
        (review_pool["tool_correctness_score"] - 1.0).abs(),
        (review_pool["argument_correctness_score"] - 0.7).abs(),
        (review_pool["answer_quality_score"] - 0.7).abs(),
    ],
    axis=1,
).min(axis=1, skipna=True)

passed_pick = review_pool[review_pool["overall_pass"]].head(1)
failed_pick = review_pool[~review_pool["overall_pass"]].head(1)
reserved_ids = set(passed_pick["case_id"]) | set(failed_pick["case_id"])
borderline_pick = (
    review_pool[~review_pool["case_id"].isin(reserved_ids)]
    .sort_values("borderline_distance", na_position="last")
    .head(1)
)
manual_review = (
    pd.concat([passed_pick, borderline_pick, failed_pick], ignore_index=True)
    .drop_duplicates("case_id")
    .head(3)
)

display(
    manual_review[
        [
            "case_id",
            "input",
            "reference_context",
            "actual_output",
            "answer_quality_score",
            "answer_quality_reason",
        ]
    ]
)

## 19 — Save detailed results

This creates one new evaluation artifact. It does not overwrite upstream analytical data or model artifacts.

In [ ]:
output_path = Path(PROJECT_ROOT) / "data" / "processed" / "agentic_evaluation_results.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False)
print("Saved evaluation results to:", output_path.resolve())

## 20 — Interpretation

The Gemma agent is evaluated at three levels:

1. **Tool Correctness** evaluates whether the agent selected the expected analytical tool trajectory.
2. **Argument Correctness** evaluates whether the parameters generated for those tool calls were appropriate for the user's request.
3. **RM Answer Quality** evaluates whether the final response is grounded in reference analytical tool outputs, answers the user's question, follows the merchant-acquisition business rules, and communicates the result clearly for an RM.

Claude is used only as an independent LLM-as-a-Judge for semantic evaluation. The underlying merchant analytics, acquisition priority, product recommendation, graph values, and economics remain generated by the existing deterministic/model pipeline.

This evaluation does not prove the agent is universally correct. It measures performance only on the curated evaluation scenarios defined in this notebook. The results should not, by themselves, be used to label the model production-ready, enterprise-ready, or safe.